Attempting to emulate the structure of FActScore:

1. Distill article into atomic fact statements and a list of topics covered

2. Collect a series of k sources using Google search API score them according to reliability on a separate model then use scores to weight the influence of these outside sources on model inference

3. Feed encoded articles into a neural net to train for predicting and classifying articles

In [1]:
# Step 1 distill input into atomic facts

from sentence_transformers import SentenceTransformer
from torch import cosine_similarity
import networkx as nx

def extract_facts(text):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    sentences = text.split('.')
    facts = []
    for sentence in sentences:
        if sentence.strip():
            embedding = model.encode(sentence)
            facts.append((sentence.strip(), embedding))
    return facts

# Step 2: Generate a knowledge graph from the facts

def build_knowledge_graph(facts):
    G = nx.Graph()
    for fact, embedding in facts:
        G.add_node(fact, embedding=embedding)
    # For simplicity, we connect facts that have similar embeddings
    for i in range(len(facts)):
        for j in range(i + 1, len(facts)):
            if cosine_similarity(facts[i][1], facts[j][1]) > 0.8:  # Threshold for similarity
                G.add_edge(facts[i][0], facts[j][0])
    return G

Capture Sources for Each fact in graph

In [2]:
# Use google search API to find top k relevant documents for a query

import requests

def google_search(query, api_key, cse_id, num_results=10):
    url = f"https://www.googleapis.com/customsearch/v1?q={query}&key={api_key}&cx={cse_id}&num={num_results}"
    response = requests.get(url)
    results = response.json().get('items', [])
    return results

# evaluate the relevance of retrieved documents to each other to find most commonly agreed upon facts regardless of source bias

def evaluate_relevance(documents):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = [model.encode(doc['snippet']) for doc in documents]
    relevance_scores = []
    for i in range(len(embeddings)):
        score = 0
        for j in range(len(embeddings)):
            if i != j:
                score += cosine_similarity(embeddings[i], embeddings[j])
        relevance_scores.append((documents[i]['link'], score))
    relevance_scores.sort(key=lambda x: x[1], reverse=True)
    return relevance_scores

In [ ]:
# use external embeddings + reliability scores to rank documents and extract most relevant facts

def rank_documents(documents, relevance_scores, reliability_scores):
    ranked_docs = []
    for doc in documents:
        link = doc['link']
        relevance_score = next((score for lnk, score in relevance_scores if lnk == link), 0)
        reliability_score = reliability_scores.get(link, 0)
        overall_score = relevance_score * 0.7 + reliability_score * 0.3  # Weighted score
        ranked_docs.append((doc, overall_score))
    ranked_docs.sort(key=lambda x: x[1], reverse=True)
    return ranked_docs

# use the ranked documents to extract the most relevant facts and update the knowledge graph

def update_knowledge_graph(G, ranked_docs):
    for doc, score in ranked_docs:
        if score > 0.5:  # Threshold for relevance
            facts = extract_facts(doc['snippet'])
            for fact, embedding in facts:
                G.add_node(fact, embedding=embedding)
                for existing_fact in G.nodes:
                    if existing_fact != fact and cosine_similarity(embedding, G.nodes[existing_fact]['embedding']) > 0.8:
                        G.add_edge(fact, existing_fact)
    return G

# Use the knowledge graph to estimate the accuracy of the facts and provide a confidence score for each fact

def estimate_fact_accuracy(G):
    fact_confidence = {}
    for node in G.nodes:
        neighbors = list(G.neighbors(node))
        confidence_score = len(neighbors) / (len(G.nodes) - 1)  # Simple confidence based on connectivity
        fact_confidence[node] = confidence_score
    return fact_confidence

# use confidence scores to filter out low confidence facts and provide a final summary of the most reliable information

def summarize_facts(fact_confidence, threshold=0.5):
    reliable_facts = [fact for fact, confidence in fact_confidence.items() if confidence > threshold]
    summary = "Summary of Reliable Facts:\n" + "\n".join(reliable_facts)
    return summary


In [4]:
# train neural net on fake news dataset using defined functions to extract features and external source info to improve accuracy

import torch

def train_model(dataset, G):
    model = torch.nn.Sequential(
        torch.nn.Linear(768, 128),
        torch.nn.ReLU(),
        torch.nn.Linear(128, 1),
        torch.nn.Sigmoid()
    )
    criterion = torch.nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(10):
        for text, label in dataset:
            facts = extract_facts(text)
            features = []
            for fact, embedding in facts:
                confidence = estimate_fact_accuracy(G).get(fact, 0)
                features.append(embedding * confidence)  # Weight embedding by confidence
            if features:
                features = torch.tensor(features).mean(dim=0)  # Average features
                output = model(features)
                loss = criterion(output, torch.tensor([label], dtype=torch.float))
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
    return model

# store the trained model

import joblib

def save_model(model, filename):
    joblib.dump(model, filename)

def load_model(filename):
    return joblib.load(filename)

In [5]:
# test the model on new data and evaluate its performance

import os
import pandas as pd

df = pd.read_csv(os.path.abspath("data/fake_news_dataset.csv"))
print(df.head())

                                  title  \
0               Foreign Democrat final.   
1   To offer down resource great point.   
2          Himself church myself carry.   
3                  You unit its should.   
4  Billion believe employee summer how.   

                                                text        date    source  \
0  more tax development both store agreement lawy...  2023-03-10  NY Times   
1  probably guess western behind likely next inve...  2022-05-25  Fox News   
2  them identify forward present success risk sev...  2022-09-01       CNN   
3  phone which item yard Republican safe where po...  2023-02-07   Reuters   
4  wonder myself fact difficult course forget exa...  2023-04-03       CNN   

                 author    category label  
0          Paula George    Politics  real  
1           Joseph Hill    Politics  fake  
2        Julia Robinson    Business  fake  
3  Mr. David Foster DDS     Science  fake  
4         Austin Walker  Technology  fake  


In [8]:
# make train test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)
